In [1]:
import os
model_to_test = 'jet_baseline'
model_revision = 'the_baseline'
hls4ml_revision = 'VitisUnified'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, str(model_revision))
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration

- **Model Name**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Backend**: VitisUnified
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Architecture**: Functional model with 3 dense layers (64→32→32) + output layer
- **Dataset**: HLS4ML LHC Jets
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import os
import keras
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

2026-05-06 00:49:41.544449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778021381.592135   88250 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778021381.614931   88250 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 00:49:41.764042: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
X_train_val = np.load('X_train_val.npy')
X_test = np.load('X_test.npy')
y_train_val = np.load('y_train_val.npy')
y_test = np.load('y_test.npy')
classes = np.load('classes.npy', allow_pickle=True)

In [3]:

inputs = keras.layers.Input(shape=(16,), name='input_layer')
x = keras.layers.Dense(64, activation='relu', name='dense_0')(inputs)
x = keras.layers.Dense(32, activation='relu', name='dense_1')(x)
x = keras.layers.Dense(32, activation='relu', name='dense_2')(x)
outputs = keras.layers.Dense(5, name='dense_3')(x)

model = keras.Model(inputs=inputs, outputs=outputs)

opt = keras.optimizers.Adam(learning_rate=5e-3)

model.compile(opt, loss=['categorical_crossentropy'], metrics=['accuracy'])

I0000 00:00:1778021390.839293   88250 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5124 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [ ]:
model.summary()

# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

In [ ]:
import os
from keras.callbacks import EarlyStopping, ReduceLROnPlateau


callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=6, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=2, min_lr=1e-6, verbose=1),
]

Epoch 1/10
487/487 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3286 - loss: 1.5115
Epoch 1: val_loss improved from None to 1.14976, saving model to model_baseline/best_model.weights.h5
487/487 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4615 - loss: 1.3469 - val_accuracy: 0.5867 - val_loss: 1.1498 - learning_rate: 1.0000e-04
Epoch 2/10
485/487 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6110 - loss: 1.1119
Epoch 2: val_loss improved from 1.14976 to 1.03138, saving model to model_baseline/best_model.weights.h5
487/487 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6321 - loss: 1.0794 - val_accuracy: 0.6599 - val_loss: 1.0314 - learning_rate: 1.0000e-04
Epoch 3/10
481/487 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6653 - loss: 1.0151
Epoch 3: val_loss improved from 1.03138 to 0.97024, saving model to model_baseline/best_model.weights.h5
487/487 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6736 - loss: 0.9973 - val_accuracy: 0.6898 - val_loss: 0.9702 - learning_rate: 1.0000

In [ ]:
model.fit(
    X_train_val, y_train_val,
    validation_data=(X_test, y_test),
    epochs=60,
    batch_size=1024,
    callbacks=callbacks,
    shuffle=True
)

In [ ]:
score = model.evaluate(X_test_val, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
test_acc = score[1]

model_name=f"model_{model_revision}_acc={test_acc:.4f}.keras"
model.save(os.path.join(model_dir, model_name))